In [3]:

import json, functools
from openai import RateLimitError, APIConnectionError

MODEL = "openai/gpt-oss-20b"

PRODUCTS = {
    "notebook": 60,
    "pen": 10,
    "pencil": 5,
    "school bag": 500,
    "eraser": 5
}

SYSTEM = f"""ROLE: You are a friendly assistant for Campus Stationery Store.
FACTS: Product prices in rupees: {json.dumps(PRODUCTS)}
RULES: Never promise delivery times. If unsure, say I'll check with the owner.
Remember customer names and orders during the chat.
FORMAT: Under 50 words, friendly and simple English."""

def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"[LOG] {func.__name__} returned {result}")
        return result
    return wrapper

@log_call
def get_price(item: str, qty: int = 1) -> int:
    """Calculate the total price of stationery products."""
    return PRODUCTS[item.lower().strip()] * qty

history = [{"role": "system", "content": SYSTEM}]
MAX_TURNS = 8

def chat(user_text: str) -> str:
    history.append({"role": "user", "content": user_text})

    recent = [history[0]] + history[1:][-MAX_TURNS:]

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=recent,
            temperature=0.3
        )
        reply = response.choices[0].message.content

    except RateLimitError:
        reply = "Please try again in a minute."

    except APIConnectionError:
        reply = "Server connection failed. Try again."

    history.append({"role": "assistant", "content": reply})
    return reply

print("Total for 5 notebooks:", get_price("notebook", 5))

while True:
    text = input("You: ")

    if text.lower().strip() in ("bye", "exit", "quit"):
        print("Bot: Thank you for visiting Campus Stationery Store!")
        break

    print("Bot:", chat(text))


[LOG] get_price returned 300
Total for 5 notebooks: 300
You: bye
Bot: Thank you for visiting Campus Stationery Store!


In [10]:
!pip install -q openai

from openai import OpenAI
from getpass import getpass

key = getpass("Enter your Groq API key: ")

client = OpenAI(
    api_key=key,
    base_url="https://api.groq.com/openai/v1"
)

try:
    models = client.models.list()
    print("API connection successful!")
except Exception as e:
    print("Connection failed:", e)



Enter your Groq API key: ··········
API connection successful!


In [11]:
PRODUCTS = {
    "notebook": 60,
    "pen": 10,
    "pencil": 5,
    "school bag": 500,
    "eraser": 5
}

SYSTEM = """
You are a helpful assistant for Campus Stationery Store.

RULES:
1. Greet customers politely.
2. Answer questions about stationery products.
3. Give product prices.
4. Help customers calculate order totals.
5. Remember customer details during the conversation.
6. Do not invent product prices.
"""

def get_price(item: str, qty: int = 1) -> int:
    """Calculate the total price of a stationery item."""
    item = item.lower()

    if item not in PRODUCTS:
        return 0

    return PRODUCTS[item] * qty


# Test the function
total = get_price("notebook", 5)
print("Total for 5 notebooks:", total)


# Chatbot setup
history = [{"role": "system", "content": SYSTEM}]
MAX_TURNS = 8


def chat(user_message):
    global history

    history.append({"role": "user", "content": user_message})

    recent = [history[0]] + history[-MAX_TURNS * 2:]

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=recent,
            temperature=0.3
        )

        reply = response.choices[0].message.content
        history.append({"role": "assistant", "content": reply})

        return reply

    except Exception as e:
        return f"Error: {e}"


print("\nCampus Stationery Store Chatbot Ready!")

while True:
    text = input("You: ")

    if text.lower() in ["bye", "exit", "quit"]:
        print("Bot: Thank you for visiting Campus Stationery Store!")
        break

    print("Bot:", chat(text))

Total for 5 notebooks: 300

Campus Stationery Store Chatbot Ready!
You: What is my name?
Bot: I’m sorry, but I don’t have that information. Could you please let me know your name? That way I can address you properly and keep track of your details during our chat.
You: Hi, my name is Ravi
Bot: Hello, Ravi! 👋 How can I help you with your stationery needs today?
You: bye
Bot: Thank you for visiting Campus Stationery Store!
